# DTA Prediction — Full GPU Training (Deliverable 3)
**EGN6217 | Sathyadharini Srinivasan | Spring 2026**

### Steps:
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Run all cells (Runtime → Run all)
3. At the end, results + model are auto-downloaded to your machine

Expected time on T4 GPU: **~8–12 minutes** for 60 epochs

In [ ]:
# ── Cell 1: Install dependencies ───────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch-scatter', 'torch-sparse', 'torch-cluster', 'torch-geometric',
    '-f', 'https://data.pyg.org/whl/torch-2.0.0+cu118.html'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'rdkit-pypi', 'scipy', 'scikit-learn'], check=False)

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Cell 2: Download Davis dataset ─────────────────────────────────────────
import os, urllib.request

os.makedirs('data/davis', exist_ok=True)
os.makedirs('results',    exist_ok=True)

BASE = 'https://raw.githubusercontent.com/hkmztrk/DeepDTA/master/data/davis/'
for fname in ['Y', 'ligands_can.txt', 'proteins.txt']:
    dest = f'data/davis/{fname}'
    if not os.path.exists(dest):
        print(f'Downloading {fname}...')
        urllib.request.urlretrieve(BASE + fname, dest)
    else:
        print(f'{fname} already exists.')
print('Data ready.')

In [ ]:
# ── Cell 3: Full training script ───────────────────────────────────────────
import json, pickle, copy, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data, Batch
from rdkit import Chem
from rdkit.Chem import rdchem

DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE   = 64
EPOCHS       = 60
PATIENCE     = 12
LR           = 1e-3
WEIGHT_DECAY = 1e-4
SEED         = 42
MAX_PROT_LEN = 1000
torch.manual_seed(SEED)
print(f'Device: {DEVICE}')

# ── Atom features (9-dim, Refinement 2) ────────────────────────────────────
HYBRID_MAP = {
    rdchem.HybridizationType.SP: 1, rdchem.HybridizationType.SP2: 2,
    rdchem.HybridizationType.SP3: 3, rdchem.HybridizationType.SP3D: 4,
    rdchem.HybridizationType.SP3D2: 5
}
CHIRAL_MAP = {
    rdchem.ChiralType.CHI_TETRAHEDRAL_CW: 1,
    rdchem.ChiralType.CHI_TETRAHEDRAL_CCW: 2
}

def atom_features(atom):
    ri = atom.GetOwningMol().GetRingInfo()
    rings = [len(r) for r in ri.AtomRings() if atom.GetIdx() in r]
    return [
        atom.GetAtomicNum(), atom.GetDegree(), atom.GetFormalCharge(),
        int(atom.GetIsAromatic()), int(atom.IsInRing()),
        HYBRID_MAP.get(atom.GetHybridization(), 0),
        min(rings) if rings else 0,
        CHIRAL_MAP.get(atom.GetChiralTag(), 0),
        atom.GetTotalNumHs()
    ]

def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    edges = []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        edges += [[i,j],[j,i]]
    ei = torch.tensor(edges, dtype=torch.long).t().contiguous() if edges else torch.zeros((2,0), dtype=torch.long)
    return Data(x=x, edge_index=ei)

AA_TO_IDX = {aa: i+1 for i,aa in enumerate('ACDEFGHIKLMNPQRSTVWXY')}
def encode_protein(seq):
    enc = [AA_TO_IDX.get(a,0) for a in seq[:MAX_PROT_LEN]]
    enc += [0]*(MAX_PROT_LEN-len(enc))
    return torch.tensor(enc, dtype=torch.long)

# ── Dataset ─────────────────────────────────────────────────────────────────
class DTADataset(Dataset):
    def __init__(self, recs): self.recs = recs
    def __len__(self): return len(self.recs)
    def __getitem__(self, i): return self.recs[i]

def collate_fn(batch):
    gs, ps, ls = zip(*batch)
    return Batch.from_data_list(gs), torch.stack(ps), torch.stack(ls)

# ── Model (Refinements 2,3,6) ───────────────────────────────────────────────
class DrugEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(9, 64);   self.bn1 = nn.BatchNorm1d(64)
        self.conv2 = GCNConv(64, 128); self.bn2 = nn.BatchNorm1d(128)
        self.conv3 = GCNConv(128,128); self.bn3 = nn.BatchNorm1d(128)
    def forward(self, x, ei, batch):
        x = F.relu(self.bn1(self.conv1(x, ei)))
        x = F.relu(self.bn2(self.conv2(x, ei)))
        x = F.relu(self.bn3(self.conv3(x, ei)))
        return global_mean_pool(x, batch)

class ProteinEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb   = nn.Embedding(26, 128, padding_idx=0)
        self.conv1 = nn.Conv1d(128, 32, 4, padding=1)
        self.conv2 = nn.Conv1d(32,  64, 6, padding=2)
        self.conv3 = nn.Conv1d(64,  96, 8, padding=3)
    def forward(self, x):
        x = self.emb(x).permute(0,2,1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        return x.max(dim=2).values

class DTAModel_v2(nn.Module):
    def __init__(self):
        super().__init__()
        self.drug = DrugEncoder()
        self.prot = ProteinEncoder()
        self.mlp  = nn.Sequential(
            nn.Linear(224,512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512,256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256,1)
        )
    def forward(self, dg, ps):
        d = self.drug(dg.x, dg.edge_index, dg.batch)
        p = self.prot(ps)
        return self.mlp(torch.cat([d,p], dim=1)).squeeze(1)

# ── Load data ───────────────────────────────────────────────────────────────
print('Loading data...')
with open('data/davis/ligands_can.txt') as f: drugs = json.load(f)
with open('data/davis/proteins.txt')   as f: prots = json.load(f)
with open('data/davis/Y','rb')         as f: Y = np.array(pickle.load(f, encoding='latin1'))

drug_smiles  = list(drugs.values())
prot_seqs    = list(prots.values())
log_Y        = np.log10(Y)   # Refinement 1

print(f'Y: {Y.shape}  |  log10(Kd) range: {log_Y.min():.2f} – {log_Y.max():.2f}')

print('Building graphs...')
records, skip = [], 0
n_d, n_p = Y.shape
for i in range(n_d):
    g = smiles_to_graph(drug_smiles[i]) if i < len(drug_smiles) else None
    if g is None: skip += 1; continue
    for j in range(n_p):
        seq = prot_seqs[j] if j < len(prot_seqs) else ''
        records.append((g, encode_protein(seq), torch.tensor(log_Y[i,j], dtype=torch.float)))
print(f'Records: {len(records):,}  (skipped {skip})')

n = len(records)
nv, nt = int(0.1*n), int(0.1*n)
tr_ds, v_ds, te_ds = random_split(DTADataset(records), [n-nv-nt, nv, nt],
                                   generator=torch.Generator().manual_seed(SEED))
tr_dl = DataLoader(tr_ds, BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, num_workers=2, pin_memory=True)
v_dl  = DataLoader(v_ds,  BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
te_dl = DataLoader(te_ds, BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
print(f'Train {len(tr_ds):,} | Val {len(v_ds):,} | Test {len(te_ds):,}')

# ── Training ─────────────────────────────────────────────────────────────────
model = DTAModel_v2().to(DEVICE)
print(f'Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, 'min', factor=0.5, patience=5)
crit  = nn.MSELoss()

def run_epoch(dl, train=True):
    model.train() if train else model.eval()
    tot, n = 0.0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for dg, ps, lb in dl:
            dg, ps, lb = dg.to(DEVICE), ps.to(DEVICE), lb.to(DEVICE)
            pr = model(dg, ps)
            loss = crit(pr, lb)
            if train:
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            tot += loss.item()*len(lb); n += len(lb)
    return tot/n

print(f'\n{"─"*60}')
print(f'{"Epoch":>6} {"Train MSE":>10} {"Val MSE":>10} {"LR":>10} {"Time":>6}')
print(f'{"─"*60}')

best_val, patience_cnt, best_w = float('inf'), 0, None
history = {'train':[], 'val':[]}

for ep in range(1, EPOCHS+1):
    t0 = time.time()
    tl = run_epoch(tr_dl, True)
    vl = run_epoch(v_dl,  False)
    history['train'].append(tl); history['val'].append(vl)
    sched.step(vl)
    lr = opt.param_groups[0]['lr']
    star = '★' if vl < best_val else ' '
    print(f'{ep:>6} {tl:>10.4f} {vl:>10.4f} {lr:>10.2e} {time.time()-t0:>5.1f}s {star}')
    if vl < best_val:
        best_val = vl; patience_cnt = 0
        best_w = copy.deepcopy(model.state_dict())
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'\n[Early Stop] Best Val MSE: {best_val:.4f}')
            break

model.load_state_dict(best_w)
torch.save(best_w, 'results/dta_model_v2_best.pt')
print('\nCheckpoint saved.')

In [ ]:
# ── Cell 4: Evaluate & plot ─────────────────────────────────────────────────
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

model.eval()
all_p, all_l = [], []
with torch.no_grad():
    for dg, ps, lb in te_dl:
        pr = model(dg.to(DEVICE), ps.to(DEVICE))
        all_p.extend(pr.cpu().numpy())
        all_l.extend(lb.numpy())
all_p, all_l = np.array(all_p), np.array(all_l)

mse  = mean_squared_error(all_l, all_p)
rmse = float(np.sqrt(mse))
mae  = mean_absolute_error(all_l, all_p)
r, _ = pearsonr(all_l, all_p)
r2   = r2_score(all_l, all_p)

# CI on sample
idx = np.random.choice(len(all_l), min(2000, len(all_l)), replace=False)
yt, yp = all_l[idx], all_p[idx]
conc = tot_ci = 0
for i in range(len(yt)):
    for j in range(i+1, len(yt)):
        if yt[i] != yt[j]:
            tot_ci += 1
            if (yp[i]>yp[j]) == (yt[i]>yt[j]): conc += 1
ci = conc/tot_ci if tot_ci else 0

print('══════════════════════════════════════════════')
print('  TEST SET RESULTS')
print('══════════════════════════════════════════════')
print(f'  MSE            : {mse:.4f}')
print(f'  RMSE           : {rmse:.4f}')
print(f'  MAE            : {mae:.4f}')
print(f'  Pearson r      : {r:.4f}')
print(f'  R²             : {r2:.4f}')
print(f'  Concordance Idx: {ci:.4f}')
print('══════════════════════════════════════════════')

np.save('results/test_preds.npy',  all_p)
np.save('results/test_labels.npy', all_l)

import json as _j
with open('results/training_history.json','w') as f:
    _j.dump({'train_loss': history['train'], 'val_loss': history['val'],
             'metrics': {'MSE':mse,'RMSE':rmse,'MAE':mae,'Pearson_r':float(r),'R2':r2,'CI':ci}}, f, indent=2)

# Plots
plt.rcParams.update({'figure.dpi':150,'font.size':11})
fig, axes = plt.subplots(1,3, figsize=(16,4))

# Loss curves
eps = range(1, len(history['train'])+1)
axes[0].plot(eps, history['train'], label='Train', color='steelblue', lw=2)
axes[0].plot(eps, history['val'],   label='Val',   color='tomato',    lw=2)
axes[0].axhline(best_val, linestyle='--', color='green', label=f'Best={best_val:.4f}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE'); axes[0].set_title('Loss Curves')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Pred vs actual
axes[1].scatter(all_l, all_p, alpha=0.2, s=4, color='steelblue')
lm = [min(all_l.min(),all_p.min())-0.1, max(all_l.max(),all_p.max())+0.1]
axes[1].plot(lm, lm, 'r--', lw=1.5)
axes[1].set_xlabel('Actual log10(Kd)'); axes[1].set_ylabel('Predicted')
axes[1].set_title(f'Pred vs Actual  r={r:.3f}'); axes[1].grid(alpha=0.3)

# Residuals
res = all_p - all_l
axes[2].hist(res, bins=60, color='darkorange', edgecolor='white', alpha=0.85)
axes[2].axvline(0, color='red', lw=1.5, linestyle='--')
axes[2].set_xlabel('Residual'); axes[2].set_ylabel('Count')
axes[2].set_title(f'Residuals  std={res.std():.3f}'); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/all_plots.png')
plt.show()
print('Plots saved.')

In [ ]:
# ── Cell 5: Download everything ─────────────────────────────────────────────
import shutil
from google.colab import files

shutil.make_archive('dta_results', 'zip', 'results')
files.download('dta_results.zip')
print('Downloaded dta_results.zip — unzip into your DEEP-LEARNING-2/results/ folder')